In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()

data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]

columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)

df.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+



In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("orders_delta")
df.display()

id,customer_id,product,amount
1,C001,Laptop,50000
2,C002,Mobile,15000
3,C003,Tablet,20000
4,C004,Laptop,55000


In [0]:
from pyspark.sql import Row

# define columns (IMPORTANT)
columns = ["id", "customer_id", "product", "amount"]

new_data = [Row(5, "C005", "Camera", 30000)]

df_insert = spark.createDataFrame(new_data, columns)

df_insert.write.format("delta").mode("append").saveAsTable("orders_delta")
df_insert.display()

id,customer_id,product,amount
5,C005,Camera,30000


In [0]:
%sql
UPDATE orders_delta
SET amount = 18000
WHERE id = 2;

num_affected_rows
1


In [0]:
spark.sql("""
DELETE FROM orders_delta
WHERE id = 1
""")

DataFrame[num_affected_rows: bigint]

In [0]:
updates = [
    (3, "C003", "Tablet", 22000),  # update
    (6, "C006", "Watch", 8000)     # insert
]

df_updates = spark.createDataFrame(updates, columns)
df_updates.createOrReplaceTempView("updates")
df_updates.display()

id,customer_id,product,amount
3,C003,Tablet,22000
6,C006,Watch,8000


In [0]:
from pyspark.sql.functions import lit

df_new = spark.table("orders_delta").withColumn("country", lit("India"))

df_new.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("orders_delta")
df_new.display()

id,customer_id,product,amount,country
2,C002,Mobile,18000,India
3,C003,Tablet,20000,India
4,C004,Laptop,55000,India
5,C005,Camera,30000,India
5,C005,Camera,30000,India
5,C005,Camera,30000,India
5,C005,Camera,30000,India


In [0]:
spark.sql("DESCRIBE HISTORY orders_delta").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
12,2026-04-14T16:04:49.000Z,70812224371903,22pa1a0467@vishnu.edu.in,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1318377408562231),ed4d115c-26df-4e87-8ff5-0708e26096b2,0414-155437-r4dlgvv4-v2n,11,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1719, numDeletionVectorsRemoved -> 0, numOutputRows -> 7, numOutputBytes -> 1719)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-14T16:04:09.000Z,70812224371903,22pa1a0467@vishnu.edu.in,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1318377408562231),2b6cebb2-9301-4ed8-830f-27cb9d57df53,0414-155437-r4dlgvv4-v2n,10,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1492, numDeletionVectorsRemoved -> 0, numOutputRows -> 7, numOutputBytes -> 1719)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-14T16:02:19.000Z,70812224371903,22pa1a0467@vishnu.edu.in,DELETE,"Map(predicate -> [""(id#12382L = 1)""])",null,List(1318377408562231),6671cc89-f86d-4c6a-b81f-c486af2e287b,0414-155437-r4dlgvv4-v2n,9,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 540, numDeletionVectorsUpdated -> 0, numDeletedRows -> 0, scanTimeMs -> 534, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 0)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-14T16:02:04.000Z,70812224371903,22pa1a0467@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1318377408562231),3e90e797-8d1f-422e-86af-9df9dbde77f3,0414-155437-r4dlgvv4-v2n,8,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1506, p25FileSize -> 1492, numDeletionVectorsRemoved -> 1, minFileSize -> 1492, numAddedFiles -> 1, maxFileSize -> 1492, p75FileSize -> 1492, p50FileSize -> 1492, numAddedBytes -> 1492)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
8,2026-04-14T16:02:01.000Z,70812224371903,22pa1a0467@vishnu.edu.in,DELETE,"Map(predicate -> [""(id#12115L = 1)""])",null,List(1318377408562231),3e90e797-8d1f-422e-86af-9df9dbde77f3,0414-155437-r4dlgvv4-v2n,7,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1863, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1371, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 491)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-14T15:59:12.000Z,70812224371903,22pa1a0467@vishnu.edu.in,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1318377408562231),4f04f774-8af8-4480-a468-e2add5589052,0414-155437-r4dlgvv4-v2n,6,SnapshotIsolation,false,"Map(numRemovedFiles -> 6, numRemovedBytes -> 7720, p25FileSize -> 1506, numDeletionVectorsRemoved -> 1, minFileSize -> 1506, numAddedFiles -> 1, maxFileSize -> 1506, p75FileSize -> 1506, p50FileSize -> 1506, numAddedBytes -> 1506)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-04-14T15:59:09.000Z,70812224371903,22pa1a0467@vishnu.edu.in,UPDATE,"Map(predicate -> [""(id#11680L = 2)""])",null,List(1318377408562231),4f04f774-8af8-4480-a468-e2add5589052,0414-155437-r4dlgvv4-v2n,5,WriteSeria

In [0]:
df_old = spark.sql("SELECT * FROM orders_delta VERSION AS OF 1")
df_old.show()

+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+



In [0]:
spark.sql("RESTORE TABLE orders_delta TO VERSION AS OF 1").display()


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1374,1,0,0,0,0
